# U-Net with Encoders

**Задача**: сегментация зубов    
**Среда**: Google Colab

Особенности реализации:
- **Архитектура**: классическая U-Net, U-Net++
- **Энкодеры**:
  - resnet34, efficientnet-b3, densenet121, mobilenet_v2
  - imagenet, ssl, swsl, None
- **Функция потерь** Combined:
  - CrossEntropy Loss, вес = 1.0
  - Dice Loss, вес = 1.0
  - Focal Loss вес = 1.0 / 0.5
- **Аугментации** есть

**Датасет**: открытый датасет **teeth-seg-3537 Computer Vision Model** [godento2](https://universe.roboflow.com/godento2/teeth-seg-3537-iaky1), источник Roboflow.

## Установка пакетов и импорт библиотек

In [ ]:
# Установка tree
!apt-get install tree -qqq > /dev/null

In [ ]:
# Установка segmentation_models_pytorch
!pip install segmentation_models_pytorch -q

In [ ]:
import os
from google.colab import drive
import zipfile
from pathlib import Path
import yaml
import json
from tqdm import tqdm
from datetime import datetime
import random
import traceback

import cv2
from PIL import Image
import numpy as np
import argparse

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms

import segmentation_models_pytorch as smp

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from scipy import ndimage

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import seaborn as sns
# настройка визуализации
%matplotlib inline
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = "retina"

In [ ]:
# скачиваем свой модуль с необходимыми функциями из своего репозитория
!wget https://raw.githubusercontent.com/drSever/MIPT_X-RayDent/refs/heads/master/01_teeth_segmentation/03_U-Net_encoders/functions_unet_encoders.py --no-cache

In [ ]:
# импорт из нашего скрипта
import functions_unet_encoders as f
from functions_unet_encoders import (
    TeethSegmentationDataset,
    get_loss_function,
    SegmentationMetrics,
    load_checkpoint,
    plot_training_history,
    create_model
    )

## Для воспроизводимости экспериментов

In [ ]:
SEED = 42
f.set_seed(SEED)

## Получение датасета (нет детских снимков, формат масок YOLOv8)

In [ ]:
DATASET_PATH_ZIP = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/02_Datasets/Godento2v2.zip'
DATASET_PATH_UNZIP = '/content'
DATASET_DIR = '/content/dataset'

In [ ]:
drive.mount('/content/gdrive')

In [ ]:
# Распаковываем
with zipfile.ZipFile(DATASET_PATH_ZIP, 'r') as zip_ref:
    for file in tqdm(zip_ref.namelist(), desc='Распаковка архива'):
        zip_ref.extract(file, DATASET_PATH_UNZIP)

# переименовываем каталог с распакованным датасетом
os.rename('/content/Godento2v2', '/content/dataset')

print(f"\nАрхив успешно распакован в: {DATASET_DIR}")

In [ ]:
!tree {DATASET_DIR} -L 3 -d  # -d только директории, -L 3 - глубина 3 уровня

## Настройки и обучение

In [ ]:
### Настройки и параметры

DATASET_PATH = '/content/dataset'

# Сохранение результатов
RESULTS_PATH = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/04_TeethSegmentation/U-Net_encoders/'
SAVE_DIR = RESULTS_PATH + "resnet50_ssl_unetplusplus_fl_05"

# Параметры модели
ENCODER_NAME = 'resnet50'  # resnet34, efficientnet-b3, densenet121, mobilenet_v2
ENCODER_WEIGHTS = 'ssl'  # imagenet, ssl, swsl, None
ARCHITECTURE = 'unetplusplus'  # unet, unetplusplus, fpn, deeplabv3plus

# Параметры обучения
LEARNING_RATE = 1e-4
BATCH_SIZE = 8 # L4 = 8, A100 = 16, resnet50_ssl_unetplusplus 8 на A100
NUM_EPOCHS = 150
IMG_SIZE = 512
NUM_CLASSES = 33
PATIENCE = 15 # Для early stopping

# Регуляризация
DROPOUT = 0.0
WEIGHT_DECAY = 5e-4  # используем L2 регуляризация весов
LABEL_SMOOTHING = 0.0

# Loss weights
CE_WEIGHT = 1.0
DICE_WEIGHT = 1.0
FOCAL_WEIGHT = 0.5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_PIN_MEMORY = torch.cuda.is_available() # Используем pin_memory только с CUDA

In [ ]:
# Функция для определения трансформаций для обучения нашей модели
def get_train_transforms(img_size=IMG_SIZE):
    """
    Аугментации для ортопантомограмм
    Специфично для сегментации зубов на рентгеновских снимках.
    """
    return A.Compose([
        # ------------------------------------------------------------
        # 1. ГЕОМЕТРИЧЕСКИЕ ПРЕОБРАЗОВАНИЯ
        # ------------------------------------------------------------
        A.Affine(
            scale=(0.95, 1.05),               # Масштаб: случайное увеличение/уменьшение до ±5%
            translate_percent=(0.03, 0.03),   # Сдвиг по горизонтали/вертикали до 3% от размера
            rotate=(-3, 3),                   # Поворот до ±3 градусов
            p=0.5,                            # Вероятность применения 50%
            border_mode=cv2.BORDER_CONSTANT,  # Новые области заполняются константой (чёрным)
            fill=0,                           # Значение для заполнения на изображении (0 — чёрный)
            fill_mask=0                       # Значение для заполнения на маске (0 — фон)
        ),
        # ОПИСАНИЕ: лёгкие аффинные искажения имитируют небольшие повороты головы пациента,
        # разный масштаб съёмки и смещения челюсти в кадре. Все изменения незначительны,
        # чтобы не нарушить анатомическую структуру зубного ряда.

        # ------------------------------------------------------------
        # 2. МЯГКИЕ ЭЛАСТИЧНЫЕ ДЕФОРМАЦИИ (применяются редко)
        # ------------------------------------------------------------
        A.ElasticTransform(
            alpha=0.5,                            # Интенсивность сдвига пикселей (малое значение → слабая деформация)
            sigma=25,                             # Степень размытия поля деформации (сглаживает искажения)
            p=0.1,                                # Вероятность 10% — очень осторожное применение
            mask_interpolation=cv2.INTER_NEAREST  # Для масок — интерполяция без сглаживания
        ),
        # ОПИСАНИЕ: ElasticTransform моделирует незначительные неоднородности тканей или
        # естественные искривления челюсти. Параметры подобраны так, чтобы зубы оставались
        # узнаваемыми, но контуры слегка «дышали». Высокая вероятность не используется,
        # чтобы не создавать неестественных форм.

        # ------------------------------------------------------------
        # 3. КОРРЕКЦИЯ КОНТРАСТА И ЯРКОСТИ (РЕНТГЕН-СПЕЦИФИЧНЫЕ)
        # ------------------------------------------------------------
        A.CLAHE(
            clip_limit=2.0,             # Порог ограничения гистограммы (выше — сильнее контраст)
            tile_grid_size=(8, 8),      # Размер ячеек для локального выравнивания на изображении (ОПТГ)
            p=0.5
        ),
        # ОПИСАНИЕ: CLAHE (Contrast Limited Adaptive Histogram Equalization) —
        # стандартный метод для рентгеновских снимков. Улучшает локальный контраст,
        # делая более чёткими границы зубов и корней, особенно на затемнённых участках.
        # Параметры (8,8) и clip_limit=2.0 дают умеренное усиление.

        A.RandomBrightnessContrast(
            brightness_limit=0.08,      # Изменение яркости до ±8%
            contrast_limit=0.08,        # Изменение контраста до ±8%
            p=0.5
        ),
        # Описание: Имитирует вариации экспозиции при съёмке — разные аппараты,
        # настройки яркости, толщина мягких тканей.

        # ------------------------------------------------------------
        # 4. ИМИТАЦИЯ АРТЕФАКТОВ (CoarseDropout) (применяются редко)
        # ------------------------------------------------------------
        A.CoarseDropout(
            num_holes_range=(1, 2),         # Количество прямоугольных артефактов: 1 или 2
            hole_height_range=(0.02, 0.04), # Высота артефакта 2–8% от высоты изображения
            hole_width_range=(0.02, 0.04),  # Ширина артефакта 2–8% от ширины
            fill=0,                         # Заливка артефакта чёрным (0) на изображении
            fill_mask=0,                    # Заливка артефакта фоном (0) на маске
            p=0.05                          # Вероятность 5% — редкое событие
        ),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.04),
            hole_width_range=(0.02, 0.04),
            fill=128,                       # Заливка артефакта серым (128) на изображении
            fill_mask=0,
            p=0.05
        ),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.04),
            hole_width_range=(0.02, 0.04),
            fill=255,                       # Заливка артефакта белым (255) на изображении
            fill_mask=0,
            p=0.05
        ),
        # ОПИСАНИЕ: CoarseDropout заменяет случайные прямоугольные области на указанный цвет.
        # В контексте ортопантомограмм это имитирует:
        #   - артефакты движения (смазанные участки);
        #   - наложение посторонних предметов (зажимы, маркеры);
        #   - отсутствие части зуба (например, разрушенная коронка);
        #   - переэкспонированные участки, пломбы, коронки, металлические вкладки.
        # Маленькая вероятность и небольшие размеры артефактов (2–8%) предотвращают
        # чрезмерное зашумление датасета и позволяют модели научиться игнорировать
        # локальные дефекты.

        # ------------------------------------------------------------
        # 5. ДОБАВЛЕНИЕ ШУМА (GaussNoise)
        # ------------------------------------------------------------
        A.GaussNoise(std_range=(0.01, 0.04), p=0.2),
        # ОПИСАНИЕ: Добавляет гауссов шум, имитируя зернистость рентгеновской
        # плёнки или электронные шумы. Параметры std_range
        # дают умеренный уровень шума, достаточный для повышения робастности модели,
        # но не разрушающий мелкие детали (например, периодонтальную щель).

        # ------------------------------------------------------------
        # 6. НОРМАЛИЗАЦИЯ И ПРЕОБРАЗОВАНИЕ В ТЕНЗОР PyTorch
        # ------------------------------------------------------------
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

def get_val_transforms(img_size=IMG_SIZE):
    """
    Трансформации для валидационного и тестового наборов
    """
    return A.Compose([
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

### Функции для создания модели и обучения

In [ ]:
# функция обучения
def train_teeth_segmentation(
    dataset_path=DATASET_PATH,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    num_epochs=None,
    img_size=IMG_SIZE,
    num_classes=NUM_CLASSES,
    patience=PATIENCE,
    device=DEVICE,
    use_pin_memory=USE_PIN_MEMORY,
    save_dir=SAVE_DIR,
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    architecture=ARCHITECTURE,
    weight_decay=WEIGHT_DECAY,
    ce_weight=CE_WEIGHT,
    dice_weight=DICE_WEIGHT,
    focal_weight=FOCAL_WEIGHT,
    resume_checkpoint=None
    ):
    """
    Обучает модель U-Net с предобученным энкодером для сегментации зубов.

    Функция выполняет полный цикл обучения:
        - Загружает датасет, применяет аугментации.
        - Вычисляет веса классов для балансировки.
        - Создаёт модель с указанной архитектурой и энкодером.
        - Настраивает оптимизатор (AdamW) и планировщик (ReduceLROnPlateau).
        - Обучает модель, отслеживая метрики (Dice, IoU, mAP и др.).
        - Реализует раннюю остановку и сохранение лучшей модели.
        - Сохраняет историю обучения и строит графики.

    Args:
        dataset_path (str): Путь к датасету.
        learning_rate (float): Начальная скорость обучения.
        batch_size (int): Размер батча.
        num_epochs (int, optional): Количество эпох. Должно быть задано.
        img_size (int): Размер входных изображений.
        num_classes (int): Количество классов (включая фон).
        patience (int): Терпение для ранней остановки.
        device (str): Устройство ('cuda' или 'cpu').
        use_pin_memory (bool): Использовать ли pin_memory в DataLoader.
        save_dir (str): Директория для сохранения результатов.
        encoder_name (str): Название энкодера (например, 'efficientnet-b3').
        encoder_weights (str или None): Предобученные веса ('imagenet' или None).
        architecture (str): Тип архитектуры ('unet', 'unetplusplus', ...).
        weight_decay (float): Коэффициент регуляризации L2.
        ce_weight (float): Вес компоненты CrossEntropy в функции потерь.
        dice_weight (float): Вес Dice Loss.
        focal_weight (float): Вес Focal Loss.
        resume_checkpoint (str, optional): Путь к чекпоинту для продолжения обучения.

    Returns:
        tuple: (training_history, save_dir)
            - training_history (dict): История метрик по эпохам.
            - save_dir (str): Путь к директории с сохранёнными файлами.

    Raises:
        FileNotFoundError: Если dataset_path не существует.
    """

    print(f"Используется устройство: {device}")
    if torch.cuda.is_available():
        print(f"GPU память: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

    print(f"\nПараметры модели:")
    print(f"  Архитектура: {architecture}")
    print(f"  Энкодер: {encoder_name}")
    print(f"  Предобученные веса: {encoder_weights}")

    # Проверка существования датасета
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Датасет не найден по пути: {dataset_path}")

    # Трансформации для аугментации данных
    train_transform = get_train_transforms(img_size)
    val_transform = get_val_transforms(img_size)

    # Загрузка датасетов
    print("\nЗагрузка датасетов...")
    train_dataset = TeethSegmentationDataset(
        dataset_path, split='train',
        transform=train_transform, img_size=img_size
    )

    val_dataset = TeethSegmentationDataset(
        dataset_path, split='valid',
        transform=val_transform, img_size=img_size
    )

    # Адаптивное количество workers
    num_workers = min(2, len(train_dataset) // batch_size) if len(train_dataset) > batch_size else 0

    # Создание DataLoader'ов
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=num_workers, pin_memory=use_pin_memory
    )

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size,
        shuffle=False, num_workers=num_workers, pin_memory=use_pin_memory
    )

    print(f"Тренировочных образцов: {len(train_dataset)}")
    print(f"Валидационных образцов: {len(val_dataset)}")

    # Получаем веса классов для борьбы с дисбалансом
    print("\nВычисление весов классов...")
    class_weights = train_dataset.get_class_weights().to(device)
    print(f"Вес фона (класс 0): {class_weights[0]:.6f}")
    print(f"Средний вес зубов: {class_weights[1:].mean():.4f}")

    # Создание модели с предобученным энкодером
    print("\nСоздание модели...")
    model = create_model(
        architecture=architecture,
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=1,
        classes=num_classes
    ).to(device)

    # Функция потерь
    criterion = get_loss_function(
        'combined', num_classes, class_weights,
        ce_weight=ce_weight, dice_weight=dice_weight, focal_weight=focal_weight
    )

    # Оптимизатор и планировщик
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    # Метрики
    class_names = train_dataset.class_names
    train_metrics = SegmentationMetrics(num_classes, class_names, exclude_background=True)
    val_metrics = SegmentationMetrics(num_classes, class_names, exclude_background=True)

    # Инициализация переменных
    start_epoch = 0
    best_val_dice = 0.0
    epochs_without_improvement = 0

    # Создание директории для сохранения
    os.makedirs(save_dir, exist_ok=True)
    print(f"Модели будут сохранены в: {save_dir}")

    # История обучения
    training_history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'train_dice': [],
        'val_dice': [],
        'train_iou': [],
        'val_iou': [],
        'train_accuracy': [],
        'val_accuracy': [],
        'train_map50': [],
        'val_map50': [],
        'train_map50_95': [],
        'val_map50_95': [],
        'train_f1_macro': [],
        'val_f1_macro': [],
        'train_f1_micro': [],
        'val_f1_micro': [],
        'train_precision_macro': [],
        'val_precision_macro': [],
        'train_recall_macro': [],
        'val_recall_macro': [],
        'train_precision_micro': [],
        'val_precision_micro': [],
        'train_recall_micro': [],
        'val_recall_micro': [],
        'learning_rate': []
    }

    # Загрузка чекпоинта если указан
    if resume_checkpoint is not None:
        checkpoint_info = load_checkpoint(
            resume_checkpoint, model, optimizer, scheduler, device
        )
        start_epoch = checkpoint_info['start_epoch']
        best_val_dice = checkpoint_info['best_val_dice']
        if checkpoint_info['training_history'] is not None:
            training_history = checkpoint_info['training_history']

    print("\nНачинаем обучение...")

    # Цикл обучения
    for epoch in range(start_epoch, num_epochs):
        # Тренировка
        model.train()
        train_metrics.reset()
        train_loss = 0.0

        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')

        for batch_idx, (images, masks) in enumerate(train_pbar):
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            result = criterion(outputs, masks)
            if isinstance(result, tuple):
                loss, loss_dict = result
            else:
                loss = result

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_metrics.update(outputs, masks)

            train_pbar.set_postfix({
                'Loss': f"{loss.item():.4f}",
                'Avg_Loss': f"{train_loss/(batch_idx+1):.4f}"
            })

        # Валидация
        model.eval()
        val_metrics.reset()
        val_loss = 0.0

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')

            for batch_idx, (images, masks) in enumerate(val_pbar):
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)

                result = criterion(outputs, masks)
                if isinstance(result, tuple):
                    loss, _ = result
                else:
                    loss = result

                val_loss += loss.item()
                val_metrics.update(outputs, masks)

                val_pbar.set_postfix({
                    'Loss': f"{loss.item():.4f}",
                    'Avg_Loss': f"{val_loss/(batch_idx+1):.4f}"
                })

        # Вычисляем mAP метрики
        print(f"\n[Эпоха {epoch+1}] Вычисление mAP метрик...")
        train_metrics.compute_map_metrics()
        val_metrics.compute_map_metrics()

        # Вычисляем остальные метрики
        train_results = train_metrics.compute()
        val_results = val_metrics.compute()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        # Обновляем планировщик
        scheduler.step(avg_val_loss)

        # Сохраняем историю
        current_lr = optimizer.param_groups[0]['lr']
        training_history['epoch'].append(epoch + 1)
        training_history['train_loss'].append(avg_train_loss)
        training_history['val_loss'].append(avg_val_loss)
        training_history['train_dice'].append(train_results['mean_dice'])
        training_history['val_dice'].append(val_results['mean_dice'])
        training_history['train_iou'].append(train_results['mean_iou'])
        training_history['val_iou'].append(val_results['mean_iou'])
        training_history['train_accuracy'].append(train_results['pixel_accuracy'])
        training_history['val_accuracy'].append(val_results['pixel_accuracy'])
        training_history['train_map50'].append(train_results['map50'])
        training_history['val_map50'].append(val_results['map50'])
        training_history['train_map50_95'].append(train_results['map50_95'])
        training_history['val_map50_95'].append(val_results['map50_95'])
        training_history['train_f1_macro'].append(train_results['f1_macro'])
        training_history['val_f1_macro'].append(val_results['f1_macro'])
        training_history['train_f1_micro'].append(train_results['f1_micro'])
        training_history['val_f1_micro'].append(val_results['f1_micro'])
        training_history['train_precision_macro'].append(train_results['precision_macro'])
        training_history['val_precision_macro'].append(val_results['precision_macro'])
        training_history['train_recall_macro'].append(train_results['recall_macro'])
        training_history['val_recall_macro'].append(val_results['recall_macro'])
        training_history['train_precision_micro'].append(train_results['precision_micro'])
        training_history['val_precision_micro'].append(val_results['precision_micro'])
        training_history['train_recall_micro'].append(train_results['recall_micro'])
        training_history['val_recall_micro'].append(val_results['recall_micro'])
        training_history['learning_rate'].append(current_lr)

        # Выводим результаты
        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        print(f"Train Dice: {train_results['mean_dice']:.4f}, Val Dice: {val_results['mean_dice']:.4f}")
        print(f"Train IoU: {train_results['mean_iou']:.4f}, Val IoU: {val_results['mean_iou']:.4f}")
        print(f"Val mAP@0.5: {val_results['map50']:.4f}, Val mAP@0.5:0.95: {val_results['map50_95']:.4f}")
        print(f"Learning Rate: {current_lr:.2e}")

        # Early stopping и сохранение
        if val_results['mean_dice'] > best_val_dice:
            best_val_dice = val_results['mean_dice']
            epochs_without_improvement = 0

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_dice': best_val_dice,
                'train_results': train_results,
                'val_results': val_results,
                'training_history': training_history,
                'architecture': architecture,
                'encoder_name': encoder_name,
                'encoder_weights': encoder_weights
            }, os.path.join(save_dir, 'best_model.pth'))
            print(f"Сохранена лучшая модель с Dice: {best_val_dice:.4f}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping: нет улучшений {patience} эпох подряд")
            break

        print("-" * 50)

    # Сохранение истории и графиков
    history_path = os.path.join(save_dir, 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(training_history, f, indent=2)

    plot_training_history(training_history, save_dir)

    print("\nОбучение завершено!")
    print(f"Лучший Dice коэффициент: {best_val_dice:.4f}")
    print(f"Модели сохранены в: {save_dir}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return training_history, save_dir

### Процесс обучения

In [ ]:
%%time
history, save_dir = train_teeth_segmentation(
    # путь к датасету
    dataset_path=DATASET_PATH,
    # learning rate
    learning_rate=LEARNING_RATE,
    # размер батча
    batch_size=BATCH_SIZE,
    # число эпох
    num_epochs=NUM_EPOCHS,
    # размер изображения на вход
    img_size=IMG_SIZE,
    # число классов (32 зуба + фон = 33)
    num_classes=NUM_CLASSES,
    # для early stopping
    patience=PATIENCE,
    # устройство
    device=DEVICE,
    # применение use pin memory
    use_pin_memory=USE_PIN_MEMORY,
    # путь для сохранения результатов
    save_dir=SAVE_DIR,
    # энкодер
    encoder_name=ENCODER_NAME,
    # веса энкодера
    encoder_weights=ENCODER_WEIGHTS,
    # архитектура
    architecture=ARCHITECTURE,
    # используем L2 регуляризация весов
    weight_decay=WEIGHT_DECAY,
    # вес CrossEntropy Loss
    ce_weight=CE_WEIGHT,
    # вес DICE Loss
    dice_weight=DICE_WEIGHT,
    # вес Focal Loss
    focal_weight=FOCAL_WEIGHT
)

### Дообучение (при необходимости)

In [ ]:
# Дообучение
# Если первичное обучение 5 эпох, а надо дообучить еще 5, то NUM_EPOCHS = 10
NUM_EPOCHS = 10

In [ ]:
%%time
history, save_dir = train_teeth_segmentation(
    # путь к датасету
    dataset_path=DATASET_PATH,
    # learning rate
    learning_rate=LEARNING_RATE,
    # размер батча
    batch_size=BATCH_SIZE,
    # число эпох
    num_epochs=NUM_EPOCHS,
    # размер изображения на вход
    img_size=IMG_SIZE,
    # число классов (32 зуба + фон = 33)
    num_classes=NUM_CLASSES,
    # для early stopping
    patience=PATIENCE,
    # устройство
    device=DEVICE,
    # применение use pin memory
    use_pin_memory=USE_PIN_MEMORY,
    # путь для сохранения результатов
    save_dir=SAVE_DIR,
    # энкодер
    encoder_name=ENCODER_NAME,
    # веса энкодера
    encoder_weights=ENCODER_WEIGHTS,
    # архитектура
    architecture=ARCHITECTURE,
    # используем L2 регуляризация весов
    weight_decay=WEIGHT_DECAY,
    # вес CrossEntropy Loss
    ce_weight=CE_WEIGHT,
    # вес DICE Loss
    dice_weight=DICE_WEIGHT,
    # вес Focal Loss
    focal_weight=FOCAL_WEIGHT

    # загрузка чекпоинта для дообучения
    resume_checkpoint=f'{SAVE_DIR}/best_model.pth'
)

## История обучения

In [ ]:
# путь для сохранения графиков истории обучения
output_dir = SAVE_DIR
# путь к файлу с историей обучения
history_path = SAVE_DIR + "/training_history.json"

In [ ]:
# Создаем папку для графиков
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Загружаем и визуализируем
print("Загрузка истории обучения...")
history = f.load_training_history(history_path)

f.print_training_summary(history)

print("Создание графиков...")
fig = f.plot_metrics_comparison(history, output_dir, show_plots = True)
plt.close(fig)
print(f"Графики сохранены в: {output_dir}")

## Инференс на тестовой подвыборке

In [ ]:
# Параметры
MODEL_PATH = SAVE_DIR + '/best_model.pth'
TEST_SAVE_DIR = SAVE_DIR + '/test_results'

In [ ]:
# Создаем директорию для результатов
os.makedirs(TEST_SAVE_DIR, exist_ok=True)

# Загружаем модель
print(f"Используется устройство: {DEVICE}")
print(f"Загрузка модели из: {MODEL_PATH}")
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

# Получаем параметры модели из чекпоинта
architecture = checkpoint.get('architecture', ARCHITECTURE)
encoder_name = checkpoint.get('encoder_name', ENCODER_NAME)
encoder_weights = checkpoint.get('encoder_weights', None)  # None для загрузки весов из чекпоинта

# Создаем модель
model = create_model(
    architecture=architecture,
    encoder_name=encoder_name,
    encoder_weights=encoder_weights,
    in_channels=1,
    classes=NUM_CLASSES
).to(DEVICE)

# Загружаем веса
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Модель успешно загружена с устройства: {DEVICE}")

# Загружаем тестовый датасет
print(f"Загрузка test датасета из: {DATASET_PATH}")
test_transform = get_val_transforms(IMG_SIZE)

test_dataset = TeethSegmentationDataset(
    DATASET_PATH, split='test',
    transform=test_transform, img_size=IMG_SIZE
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=USE_PIN_MEMORY
)

print(f"Количество test образцов: {len(test_dataset)}")

# Запускаем инференс
test_results = f.run_inference(
    model=model,
    test_loader=test_loader,
    device=DEVICE,
    num_classes=NUM_CLASSES,
    class_names=test_dataset.class_names,
    save_dir=TEST_SAVE_DIR
)

## Инференс на ОПТГ

### Инференс на одном изображении

In [ ]:
## Параметры

# путь к обученной модели
MODEL_PATH = SAVE_DIR + '/best_model.pth'

# путь к изображению для инференса
IMAGE_PATH = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/01_Data/RealTest/№9.JPG'

# путь к data.yaml датасета (для получения имен классов)
DATA_YAML_PATH = DATASET_PATH + '/data.yaml'

# директория для сохранения результатов
OUTPUT_DIR = SAVE_DIR + '/inference_single'

In [ ]:
# Трансформации для инференса (без аугментаций)
inference_transform = get_val_transforms(IMG_SIZE)

# Запускаем инференс
mask, result_image = f.inference_single_image(
    model_path=MODEL_PATH,
    image_path=IMAGE_PATH,
    data_yaml_path=DATA_YAML_PATH,
    transform=inference_transform,
    output_dir=OUTPUT_DIR,
    device=DEVICE
)

### Инференс на пакете изображений

In [ ]:
### Инференс на пакете изображений

## Параметры

# путь к обученной модели
MODEL_PATH = SAVE_DIR + '/best_model.pth'

# путь к директории с изображениями
IMAGES_DIR = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/01_Data/RealTest/'

# путь к data.yaml датасета
DATA_YAML_PATH = DATASET_PATH + '/data.yaml'

# директория для сохранения результатов
OUTPUT_DIR = SAVE_DIR + '/inference_batch'

In [ ]:
# Трансформации для инференса
inference_transform = get_val_transforms(IMG_SIZE)

# Запускаем инференс на всех изображениях
masks, results = f.inference_multiple_images(
    model_path=MODEL_PATH,
    image_dir=IMAGES_DIR,
    data_yaml_path=DATA_YAML_PATH,
    transform=inference_transform,
    output_dir=OUTPUT_DIR,
    device=DEVICE
)

## Сохранение зависимостей

In [ ]:
!pip list --format=freeze > requirements.txt